# Student Name: Zohain Javed
# Student ID: 40457802
## Section 2: Feature Engineering
This notebook extracts 16 numerical features from the 112 hand-drawn images to build the dataset.

- **Features 1-7:** Count total black pixels and how many rows/columns have specific pixel counts (1, 2, or 3+).
- **Features 8-12:** Calculate the height, width, aspect ratio, and the maximum number of pixels in any single row or column.
- **Features 13-15:** Use `scipy.ndimage` to find connected regions and enclosed white spaces ("Eyes"), and the "Hollowness" ratio.
- **Feature 16 (Custom):** A "Roughness" ratio that counts stray pixels (those with 0 orthogonal neighbors and 1 or fewer diagonal neighbors) to see how "messy" the drawing is.

The script processes all CSV files from the `./images` directory and exports a compiled dataset to `40457802_features.csv`, sorted alphabetically by label and then by image index.

In [ ]:
# Setting up the required tools 
import os
import numpy as np
import pandas as pd
from scipy.ndimage import label as clabel

IMAGE_DIR = "./images" 
STUDENTN = "40457802"

In [ ]:
def extract_features(matrix, label, index):

    # Finding the coordinates/indices of the black pixels (0s) in the matrix
    # It is stored in the format: ([row1, row2, ...], [col1, col2, ...]) where (row1, col1) is a black pixel in the matrix
    black_pixels = np.where(matrix == 0)
    rows, cols = black_pixels
    

    # Feature 1: nr_pix (Total number of black pixels)

    nr_pix = len(black_pixels[0])
    
    # Calculating row and column sums (required for Features 2-7)
    row_counts = np.sum(matrix == 0, axis=1) # List containing number of pixels in each of the 45 rows
    col_counts = np.sum(matrix == 0, axis=0) # List containing number of pixels in each of the 45 columns
    

    # Features 2-7 (Row/Col count with n pixels)

    rows_with_1 = np.sum(row_counts == 1)
    cols_with_1 = np.sum(col_counts == 1)
    rows_with_2 = np.sum(row_counts == 2)
    cols_with_2 = np.sum(col_counts == 2)
    rows_with_3p = np.sum(row_counts >= 3)
    cols_with_3p = np.sum(col_counts >= 3)
    

    # Features 8-12 (Geometry)

    if nr_pix > 0: # Ensuring the image is not blank

        height = np.max(rows) - np.min(rows)
        width = np.max(cols) - np.min(cols)
        aspect_ratio = width / height if height > 0 else 0
        maxrow = np.max(row_counts) # Max number of pixels in any row
        maxcol = np.max(col_counts) # Max number of pixels in any column
    else:
        height = width = aspect_ratio = maxrow = maxcol = 0
        

    # Feature 13: Connected Areas (8-connectivity)

    # clabel returns the number of connected regions (unique objects), and a labeled version of the image
    # The labeled image is not needed, so we ignore it with _
    _, connected_areas = clabel(matrix == 0, structure=np.ones((3,3)))
    # structure=np.ones((3,3)) tells the function to use 8-connectivity
    # since np.ones((3,3)) has 1 in all directions to the center (8 neighbours)
    

    # Feature 14: Eyes (White regions surrounded by black)

    # White pixels are orthogonally connected (4-connectivity)
    white_struct = [[0,1,0], [1,1,1], [0,1,0]] # Defining the matrix for 4-connectivity (four neighbouring 1s to the center)

    # Labeling every separate white region
    labeled_white, num_white_areas = clabel(matrix == 1, structure=white_struct)
    
    # Find which labels are "touching" the very edge of the 45x45 canvas
    # We subtract these regions as they are background pieces, not eyes
    edge_labels = set() # Regions that are not to be counted, using set() to ignore duplicates
    edge_labels.update(labeled_white[0, :])   # Top row
    edge_labels.update(labeled_white[-1, :])  # Bottom row
    edge_labels.update(labeled_white[:, 0])   # Left col
    edge_labels.update(labeled_white[:, -1])  # Right col
    
    # Removing '0' from edge_labels if it exists (0 is the label for black pixels)
    if 0 in edge_labels:
        edge_labels.remove(0)
    
    # Eyes = Total white regions - Background regions touching the edge
    eyes = num_white_areas - len(edge_labels)
    

    # Feature 15: Hollowness

    # Identifying the pixels that belong to eyes (white pixels that aren't background)
    # ~np.isin(...) excludes all the pixels whose label is in our background set 
    is_eye_pixel = (matrix == 1) & (~np.isin(labeled_white, list(edge_labels)))
    
    # Summing the number of eye pixels
    nr_eye_pix = np.sum(is_eye_pixel)
    
    # 3. Calculating ratio: (White pixels in eyes) / (Black pixels)
    hollowness = nr_eye_pix / nr_pix if nr_pix > 0 else 0
    

    # Feature 16: Custom (Roughness/Sharpness)

    # We will define rules to check whether a pixel is considered stray or not based on the neighbours
    # The number of stray pixels can be used to define how rough the image is
    stray_pixels = 0
    
    if nr_pix > 0:
        # Checking each pixel if it is stray
        for r, c in zip(rows, cols):
            ortho = 0 # number of orthogonal neighbours, up/down/left/right
            diag = 0 # number of diagonal neighbours
            
            # Checking Orthogonal neighbors
            # We check Up, Down, Left, Right
            for o_row, o_col in [(-1,0), (1,0), (0,-1), (0,1)]:
                nbr_r, nbr_c = r + o_row, c + o_col
                if 0 <= nbr_r < 45 and 0 <= nbr_c < 45: # Staying on the canvas
                    if matrix[nbr_r, nbr_c] == 0:
                        ortho += 1
            
            # Checking Diagonal neighbors
            # We check the 4 corners
            for d_row, d_col in [(-1,-1), (-1,1), (1,-1), (1,1)]:
                nbr_r, nbr_c = r + d_row, c + d_col
                if 0 <= nbr_r < 45 and 0 <= nbr_c < 45: # Staying on the canvas
                    if matrix[nbr_r, nbr_c] == 0:
                        diag += 1
            

            # Checking if the pixel is stray
            if (ortho + diag) <= 1:
                stray_pixels += 1

    roughness = stray_pixels / nr_pix if nr_pix > 0 else 0
    


    # Returning a dictionary so it's easy to turn into a table
    return {
        'label': label,
        'Index': index,
        'nr_pix': nr_pix,
        'rows_with_1': rows_with_1,
        'cols_with_1': cols_with_1,
        'rows_with_2': rows_with_2,
        'cols_with_2': cols_with_2,
        'rows_with_3p': rows_with_3p,
        'cols_with_3p': cols_with_3p,
        'height': height,
        'width': width,
        'aspect_ratio': aspect_ratio,
        'maxrow': maxrow,
        'maxcol': maxcol,
        'connected_areas': connected_areas,
        'eyes': eyes,
        'hollowness': hollowness,
        'custom': roughness
    }

In [19]:
# Exporting the data from the features into a CSV table format

all_features = [] # List to hold all feature dictionaries

print("Starting feature extraction...")

# Reading every file in the folder
for filename in os.listdir(IMAGE_DIR):
    # Only process CSV files, ignore the PGM files
    if filename.endswith(".csv"):
        file_path = os.path.join(IMAGE_DIR, filename)
        
        # We split by '_' to grab the label and index
        parts = filename.split('_')
        
        # parts[0] is the student number, parts[1] is the label, parts[2] is the index
        label = parts[1]
        
        # Stripping off '.csv' at end of the index part
        index = parts[2].replace('.csv', '')
        
        # Loading the 45x45 matrix
        matrix = np.loadtxt(file_path, delimiter=',')
        
        # Run your extraction function
        features = extract_features(matrix, label, index)
        
        # Add the resulting dictionary to our list
        all_features.append(features)

# Converting the list of dictionaries into a Pandas DataFrame
df = pd.DataFrame(all_features)

# Sorting the data: first by label (alphabetically), then by index
df = df.sort_values(by=['label', 'Index'])

output_filename = f"{STUDENTN}_features.csv"

# Exporting to CSV without the Pandas row indices, forcing utf-8 encoding (for safety)
df.to_csv(output_filename, index=False, encoding='utf-8')

print(f"\nSucessfully Saved to {output_filename}")

Starting feature extraction...

Sucessfully Saved to 40457802_features.csv
